# ResNet50-UNet Training Notebook

Self-contained notebook for training ResNet50-UNet for semantic segmentation.

Supports both generic datasets and Cityscapes format. This notebook includes all necessary code without external imports.

In [7]:
!pip install torch torchvision tqdm pandas numpy kagglehub 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 45.7 MB/s eta 0:00:00a 0:00:01


In [2]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("electraawais/cityscape-dataset")

# print("Path to dataset files:", path)

100%|██████████| 11.0G/11.0G [05:38<00:00, 35.0MB/s]  

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2


## Configuration

In [3]:
# Training Configuration
config = {
    'data_dir': 'data',  # Directory containing train/val subdirs (or Cityscapes root dir)
    'num_classes': 19,  # Cityscapes has 19 classes (0-18), plus void class 255
    'ignore_index': 255,  # For Cityscapes void class (set to None for generic datasets)
    'img_size': 512,     # Input image size (square)
    'batch_size': 8,
    'epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'seed': 42,
    'device': 'auto',    # 'auto', 'cuda', or 'cpu'
    'use_amp': True,     # Use automatic mixed precision
    'output_dir': '/'
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

Configuration:
  data_dir: data
  num_classes: 19
  ignore_index: 255
  img_size: 512
  batch_size: 8
  epochs: 50
  learning_rate: 0.0001
  weight_decay: 1e-05
  seed: 42
  device: auto
  use_amp: True
  output_dir: /kaggle/working/


## Imports and Setup

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from PIL import Image
import numpy as np
import random
from pathlib import Path
from tqdm import tqdm
import json
import csv
from datetime import datetime
import os

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(config['seed'])

# Device setup
def get_device(device_str):
    if device_str == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_str)

device = get_device(config['device'])
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.9.0+cu128


## Model Definition

In [9]:
class DecoderBlock(nn.Module):
    """UNet decoder block with upsampling and convolution."""

    def __init__(self, in_channels, out_channels, skip_channels=0):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNetUNet(nn.Module):
    """ResNet50-UNet for semantic segmentation."""

    def __init__(self, num_classes, pretrained=True):
        super().__init__()

        # Load ResNet50 encoder
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        resnet = resnet50(weights=weights)

        # Encoder layers
        self.encoder_conv1 = resnet.conv1
        self.encoder_bn1 = resnet.bn1
        self.encoder_relu = resnet.relu
        self.encoder_maxpool = resnet.maxpool
        self.encoder_layer1 = resnet.layer1  # 256 channels
        self.encoder_layer2 = resnet.layer2  # 512 channels
        self.encoder_layer3 = resnet.layer3  # 1024 channels
        self.encoder_layer4 = resnet.layer4  # 2048 channels

        # Decoder
        self.decoder4 = DecoderBlock(2048, 1024, skip_channels=1024)  # layer4 + layer3
        self.decoder3 = DecoderBlock(1024, 512, skip_channels=512)    # decoder4 + layer2
        self.decoder2 = DecoderBlock(512, 256, skip_channels=256)     # decoder3 + layer1
        self.decoder1 = DecoderBlock(256, 128, skip_channels=64)      # decoder2 + conv1

        # Final segmentation head
        self.final_conv = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        conv1 = self.encoder_conv1(x)
        conv1 = self.encoder_bn1(conv1)
        conv1 = self.encoder_relu(conv1)

        pool1 = self.encoder_maxpool(conv1)
        layer1 = self.encoder_layer1(pool1)  # [B, 256, H/4, W/4]
        layer2 = self.encoder_layer2(layer1)  # [B, 512, H/8, W/8]
        layer3 = self.encoder_layer3(layer2)  # [B, 1024, H/16, W/16]
        layer4 = self.encoder_layer4(layer3)  # [B, 2048, H/32, W/32]

        # Decoder with skip connections
        dec4 = self.decoder4(layer4, layer3)  # [B, 1024, H/16, W/16]
        dec3 = self.decoder3(dec4, layer2)    # [B, 512, H/8, W/8]
        dec2 = self.decoder2(dec3, layer1)    # [B, 256, H/4, W/4]
        dec1 = self.decoder1(dec2, conv1)     # [B, 128, H/2, W/2]

        # Final prediction (logits)
        out = self.final_conv(dec1)  # [B, num_classes, H/2, W/2]

        # Upsample to original resolution
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)

        return out


def create_resnet_unet(num_classes, pretrained=True):
    """Factory function to create ResNet50-UNet model."""
    return ResNetUNet(num_classes, pretrained)


# Create model
model = create_resnet_unet(config['num_classes'])
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 220MB/s]


Total parameters: 73,583,315
Trainable parameters: 73,583,315


## Dataset and Data Loading

In [ ]:
CITYSCAPES_LABEL_TO_TRAINID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7,
    21: 8, 22: 9, 23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18
}


class SegmentationDataset(Dataset):
    """Dataset for semantic segmentation with images and masks."""

    def __init__(self, image_dir, mask_dir, image_size=None, augment=False, ignore_index=None, dataset_type='generic'):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.image_size = image_size
        self.augment = augment
        self.ignore_index = ignore_index

        # Find matching image/mask pairs
        self.dataset_type = dataset_type
        
        # Find matching image/mask pairs
        self.image_paths = []
        self.mask_paths = []
        
        if dataset_type == 'cityscapes':
            self._load_cityscapes_pairs()
        else:
            self._load_generic_pairs()
        
        print(f"Found {len(self.image_paths)} image-mask pairs")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path)
        mask = np.array(mask)

        # Apply augmentations if enabled
        if self.augment:
            image, mask = self._apply_augmentations(image, mask)
        
        if self.dataset_type == 'cityscapes':
            # Map labelIds → trainIds
            remapped_mask = np.full_like(mask, fill_value=255, dtype=np.uint8)
            for k, v in CITYSCAPES_LABEL_TO_TRAINID.items():
                remapped_mask[mask == k] = v
            mask = remapped_mask


        # Resize if specified
        if self.image_size is not None:
            image = TF.resize(image, self.image_size, interpolation=T.InterpolationMode.BILINEAR)
            mask_pil = Image.fromarray(mask)
            mask_pil = TF.resize(mask_pil, self.image_size, interpolation=T.InterpolationMode.NEAREST)
            mask = np.array(mask_pil)

        # Convert to tensors
        image = TF.to_tensor(image)
        mask = torch.from_numpy(mask).long()

        
        # Handle ignore_index and clamp invalid class indices
        
        
        #mask = torch.from_numpy(mask).long()
        if self.ignore_index is not None:
            mask[mask == self.ignore_index] = self.ignore_index  # keep 255 as is
        else:
            mask = torch.clamp(mask, 0, 18)


        return {
            'image': image,
            'mask': mask,
            'image_path': str(image_path),
            'mask_path': str(mask_path)
        }

    def _apply_augmentations(self, image, mask):
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = np.flip(mask, axis=1)

        if random.random() > 0.5:
            image = TF.vflip(image)
            mask = np.flip(mask, axis=0)

        rotations = [0, 90, 180, 270]
        rotation = random.choice(rotations)
        if rotation > 0:
            image = TF.rotate(image, rotation)
            mask = np.rot90(mask, k=rotation//90)

        return image, mask

    def _load_generic_pairs(self):
        """Load image-mask pairs for generic dataset format."""
        image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
        mask_extensions = {'.png', '.tiff'}
        
        # Get all image files
        for ext in image_extensions:
            self.image_paths.extend(self.image_dir.glob(f'**/*{ext}'))
        
        # Get all mask files
        for ext in mask_extensions:
            self.mask_paths.extend(self.mask_dir.glob(f'**/*{ext}'))
        
        # Sort for consistent ordering
        self.image_paths.sort()
        self.mask_paths.sort()
        
        # Verify matching pairs
        image_names = {p.stem for p in self.image_paths}
        mask_names = {p.stem for p in self.mask_paths}
        
        if image_names != mask_names:
            missing_masks = image_names - mask_names
            missing_images = mask_names - image_names
            raise ValueError(
                f"Image-mask mismatch! Missing masks for: {missing_masks}, "
                f"Missing images for: {missing_images}"
            )

    def _load_cityscapes_pairs(self):
        """Load image-mask pairs for Cityscapes dataset format."""
        # Handle both directory structures:
        # 1. Standard: leftImg8bit and gtFine at same level
        # 2. User's case: separate paths with different base directories
        
        # Find all image files
        self.image_paths = list(self.image_dir.glob('**/*_leftImg8bit.png'))
        
        # Create corresponding mask paths
        for img_path in self.image_paths:
            # Convert image path to mask path
            img_str = str(img_path)
            mask_str = img_str.replace(str(self.image_dir), str(self.mask_dir))
            mask_str = mask_str.replace('_leftImg8bit.png', '_gtFine_labelIds.png')
            mask_path = Path(mask_str)
            self.mask_paths.append(mask_path)
        
        # Verify all mask files exist
        missing_masks = []
        for mask_path in self.mask_paths:
            if not mask_path.exists():
                missing_masks.append(mask_path)
        
        if missing_masks:
            print(f"Warning: {len(missing_masks)} mask files missing")
            # Filter out missing pairs
            valid_pairs = [(img, mask) for img, mask in zip(self.image_paths, self.mask_paths) if mask.exists()]
            if valid_pairs:
                self.image_paths, self.mask_paths = zip(*valid_pairs)
            else:
                self.image_paths, self.mask_paths = [], []
        
        # Sort for consistent ordering
        if self.image_paths:
            combined = sorted(zip(self.image_paths, self.mask_paths))
            self.image_paths, self.mask_paths = zip(*combined)


## Dataset Creation

In [11]:
# Create datasets
# For Cityscapes dataset, use dataset_type="cityscapes"
# For generic dataset, use dataset_type="generic" (default)

# Example for Cityscapes:
# train_dataset = SegmentationDataset(
#     "/path/to/cityscapes/leftImg8bit/train",
#     "/path/to/cityscapes/gtFine/train",
#     image_size=(config["img_size"], config["img_size"]),
#     augment=True,
#     dataset_type="cityscapes"
#     ignore_index=config.get("ignore_index"),
# )

# Example for generic dataset:
# train_dataset = SegmentationDataset(
#     "data/train/images",
#     "data/train/masks",
#     image_size=(config["img_size"], config["img_size"]),
#     augment=True
# )

# Replace the paths below with your actual dataset paths:
data_dir = Path(config["data_dir"])

# Generic dataset structure (uncomment and modify):
# train_dataset = SegmentationDataset(
#     data_dir / "train" / "images",
#     data_dir / "train" / "masks",
#     image_size=(config["img_size"], config["img_size"]),
#     augment=True
# )
#
# val_dataset = SegmentationDataset(
#     data_dir / "val" / "images",
#     data_dir / "val" / "masks",
#     image_size=(config["img_size"], config["img_size"]),
#     augment=False
# )

# Cityscapes dataset structure (uncomment and modify):
train_dataset = SegmentationDataset(
    "/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Cityscape Dataset/leftImg8bit/train",
    "/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Fine Annotations/gtFine/train",
    image_size=(config["img_size"], config["img_size"]),
    augment=True,
    dataset_type="cityscapes"
)

val_dataset = SegmentationDataset(
    "/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Cityscape Dataset/leftImg8bit/val",
    "/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Fine Annotations/gtFine/val",
    image_size=(config["img_size"], config["img_size"]),
    augment=False,
    dataset_type="cityscapes"
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=config["batch_size"], shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset, batch_size=config["batch_size"], shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")


Found 2975 image-mask pairs
Found 500 image-mask pairs
Train samples: 2975
Val samples: 500


## Metrics

In [ ]:
class SegmentationMetrics:
    """Compute segmentation metrics: IoU, pixel accuracy, etc."""

    def __init__(self, num_classes, ignore_index=None, device=None):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.device = device or torch.device("cpu")
        self.confusion_matrix = torch.zeros((num_classes, num_classes), dtype=torch.long, device=self.device)

    def reset(self):
        self.confusion_matrix.zero_()

    def update(self, preds, targets):
        # Flatten
        if preds.dim() == 3:
            preds = preds.flatten()
        if targets.dim() == 3:
            targets = targets.flatten()

        # Align devices
        if self.confusion_matrix.device != preds.device:
            self.confusion_matrix = self.confusion_matrix.to(preds.device)

        # ✅ Apply ignore mask FIRST
        if self.ignore_index is not None:
            valid_mask = targets != self.ignore_index
            preds = preds[valid_mask]
            targets = targets[valid_mask]

        # ✅ Then clamp (only valid pixels)
        preds = torch.clamp(preds, 0, self.num_classes - 1)
        targets = torch.clamp(targets, 0, self.num_classes - 1)

        indices = self.num_classes * targets + preds
        self.confusion_matrix += torch.bincount(
            indices, minlength=self.num_classes**2
        ).reshape(self.num_classes, self.num_classes)


    def compute_iou(self):
        intersection = torch.diag(self.confusion_matrix)
        union = self.confusion_matrix.sum(dim=0) + self.confusion_matrix.sum(dim=1) - intersection
        union = torch.where(union == 0, torch.ones_like(union), union)
        class_ious = intersection.float() / union.float()
        valid_classes = union > 0
        miou = class_ious[valid_classes].mean() if valid_classes.any() else torch.tensor(0.0, device=self.confusion_matrix.device)
        return class_ious, miou

    def compute_pixel_accuracy(self):
        total_pixels = self.confusion_matrix.sum()
        correct_pixels = torch.diag(self.confusion_matrix).sum()
        return correct_pixels.float() / total_pixels.float() if total_pixels > 0 else torch.tensor(0.0, device=self.confusion_matrix.device)

    def get_metrics(self):
        class_ious, miou = self.compute_iou()
        pixel_acc = self.compute_pixel_accuracy()
        return {
            'miou': miou.item(),
            'pixel_acc': pixel_acc.item(),
            'class_ious': class_ious.tolist()
        }



def print_metrics_table(metrics):
    """Print formatted metrics table."""
    print("\n" + "="*50)
    print("SEGMENTATION METRICS")
    print("="*50)
    print(f"mIoU: {metrics['miou']:.4f}")
    print(f"Pixel Accuracy: {metrics['pixel_acc']:.4f}")
    print("\nPer-Class IoU:")
    print("-"*30)
    for i, iou in enumerate(metrics['class_ious']):
        print(f"Class {i}: {iou:.4f}")
    print("="*50)

## Training Functions

In [ ]:
# Alternative: High-contrast colormap with better color separation
def create_high_contrast_colormap(num_classes):
    """
    Create a colormap optimized for maximum contrast between adjacent classes.
    Uses a more sophisticated algorithm to ensure colors are visually distinct.
    
    Args:
        num_classes: Number of classes
        
    Returns:
        numpy array of shape (num_classes, 3) with RGB values [0-255]
    """
    colors = []
    
    # Predefined high-contrast color palette (first 20 colors)
    # These are manually selected for maximum visual distinction
    high_contrast_palette = [
        [0, 0, 0],        # 0: Black (background)
        [255, 0, 0],      # 1: Red
        [0, 255, 0],      # 2: Green
        [0, 0, 255],      # 3: Blue
        [255, 255, 0],    # 4: Yellow
        [255, 0, 255],    # 5: Magenta
        [0, 255, 255],    # 6: Cyan
        [255, 128, 0],    # 7: Orange
        [128, 0, 255],    # 8: Purple
        [255, 192, 203],  # 9: Pink
        [0, 128, 0],      # 10: Dark Green
        [128, 128, 0],    # 11: Olive
        [0, 0, 128],       # 12: Navy
        [128, 0, 0],      # 13: Maroon
        [192, 192, 192],  # 14: Silver
        [128, 128, 128],  # 15: Gray
        [255, 165, 0],    # 16: Orange Red
        [0, 255, 127],    # 17: Spring Green
        [255, 20, 147],   # 18: Deep Pink
        [70, 130, 180],   # 19: Steel Blue
    ]
    
    # Use predefined palette if we have enough colors
    if num_classes <= len(high_contrast_palette):
        colors = high_contrast_palette[:num_classes]
    else:
        # Start with predefined colors
        colors = high_contrast_palette.copy()
        
        # Generate additional colors using HSV with better spacing
        for i in range(len(high_contrast_palette), num_classes):
            # Use golden ratio for better color distribution
            golden_ratio = 0.618033988749895
            hue = (i * golden_ratio) % 1.0
            saturation = 0.6 + 0.4 * (i % 2)
            value = 0.5 + 0.5 * ((i // 2) % 2)
            
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append([int(c * 255) for c in rgb])
    
    return np.array(colors, dtype=np.uint8)


# Create both colormaps for comparison
standard_colormap = create_distinct_colormap(config['num_classes'])
high_contrast_colormap = create_high_contrast_colormap(config['num_classes'])

print("Two colormap options available:")
print("1. standard_colormap: HSV-based with good distribution")
print("2. high_contrast_colormap: Predefined high-contrast colors")
print("\nYou can use either colormap for visualization.")
print("The high_contrast_colormap is recommended for better class distinction.")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import colorsys

def create_distinct_colormap(num_classes):
    """
    Create a colormap with distinct, contrasting colors for each class.
    Uses HSV color space to maximize perceptual distance between colors.
    
    Args:
        num_classes: Number of classes
        
    Returns:
        numpy array of shape (num_classes, 3) with RGB values [0-255]
    """
    colors = []
    
    # Generate colors using HSV space for better distribution
    for i in range(num_classes):
        # Distribute hues evenly
        hue = i / num_classes
        
        # Vary saturation and value to create more distinct colors
        # Alternate between high/low saturation and brightness
        saturation = 0.7 + 0.3 * (i % 2)  # 0.7 or 1.0
        value = 0.8 + 0.2 * ((i // 2) % 2)  # 0.8 or 1.0
        
        # Convert HSV to RGB
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)
        # Convert to 0-255 range
        colors.append([int(c * 255) for c in rgb])
    
    # Ensure class 0 (background) is black for better contrast
    if num_classes > 0:
        colors[0] = [0, 0, 0]
    
    return np.array(colors, dtype=np.uint8)


def apply_colormap(mask, colormap):
    """
    Apply colormap to segmentation mask.
    
    Args:
        mask: numpy array of shape (H, W) with class indices
        colormap: numpy array of shape (num_classes, 3) with RGB colors
        
    Returns:
        numpy array of shape (H, W, 3) with RGB values [0-255]
    """
    h, w = mask.shape
    colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    # Handle ignore_index (-100) by making it gray
    valid_mask = mask >= 0
    mask_clamped = np.clip(mask, 0, len(colormap) - 1)
    
    for class_id in range(len(colormap)):
        class_pixels = (mask_clamped == class_id) & valid_mask
        colored_mask[class_pixels] = colormap[class_id]
    
    # Set ignored pixels to gray
    ignored_pixels = ~valid_mask
    colored_mask[ignored_pixels] = [128, 128, 128]
    
    return colored_mask


def visualize_prediction(image, true_mask, pred_mask, colormap, save_path=None):
    """
    Visualize original image, ground truth mask, and prediction side by side.
    
    Args:
        image: PIL Image or numpy array (H, W, 3) RGB
        true_mask: numpy array (H, W) with ground truth class indices
        pred_mask: numpy array (H, W) with predicted class indices
        colormap: numpy array (num_classes, 3) with RGB colors
        save_path: Optional path to save the visualization
    """
    # Convert inputs to numpy if needed
    if isinstance(image, Image.Image):
        image = np.array(image)
    if isinstance(true_mask, torch.Tensor):
        true_mask = true_mask.cpu().numpy()
    if isinstance(pred_mask, torch.Tensor):
        pred_mask = pred_mask.cpu().numpy()
    
    # Apply colormaps
    true_colored = apply_colormap(true_mask, colormap)
    pred_colored = apply_colormap(pred_mask, colormap)
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image', fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(true_colored)
    axes[1].set_title('Ground Truth', fontsize=14)
    axes[1].axis('off')
    
    axes[2].imshow(pred_colored)
    axes[2].set_title('Prediction', fontsize=14)
    axes[2].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    
    plt.show()


# Create colormap for Cityscapes (19 classes)
cityscapes_colormap = create_distinct_colormap(config['num_classes'])

# Display the colormap
print("Colormap for Cityscapes classes:")
print("=" * 50)
for i, color in enumerate(cityscapes_colormap):
    print(f"Class {i:2d}: RGB({color[0]:3d}, {color[1]:3d}, {color[2]:3d})")

# Visualize the colormap
fig, ax = plt.subplots(figsize=(12, 2))
ax.imshow([cityscapes_colormap], aspect='auto')
ax.set_title('Class Colormap', fontsize=14)
ax.set_xlabel('Class Index')
ax.set_xticks(range(config['num_classes']))
ax.set_xticklabels(range(config['num_classes']))
ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# Updated inference function with colormap support
def predict_single_image(image_path, model, device, img_size, colormap=None, return_colored=True):
    """
    Run inference on a single image with optional colormap.
    
    Args:
        image_path: Path to input image
        model: Trained model
        device: Device to run inference on
        img_size: Input image size
        colormap: Optional colormap array (num_classes, 3) for colored output
        return_colored: If True and colormap provided, return colored mask
    
    Returns:
        If return_colored and colormap provided: (raw_mask, colored_mask) as PIL Images
        Otherwise: raw_mask as PIL Image
    """
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    
    # Transform
    transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor()
    ])
    
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # Inference
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
    
    # Resize prediction back to original size
    pred_resized = Image.fromarray(pred.astype(np.uint8)).resize(original_size, Image.NEAREST)
    
    if return_colored and colormap is not None:
        # Apply colormap
        pred_np = np.array(pred_resized)
        colored_mask = apply_colormap(pred_np, colormap)
        colored_pil = Image.fromarray(colored_mask)
        return pred_resized, colored_pil
    
    return pred_resized


# Example: Visualize predictions on validation samples
def visualize_sample_predictions(model, dataloader, device, colormap, num_samples=3):
    """Visualize predictions on a few validation samples."""
    model.eval()
    samples_shown = 0
    
    with torch.no_grad():
        for batch in dataloader:
            if samples_shown >= num_samples:
                break
                
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            # Visualize each image in the batch
            for i in range(min(images.size(0), num_samples - samples_shown)):
                image_np = images[i].cpu().permute(1, 2, 0).numpy()
                image_np = (image_np * 255).astype(np.uint8)  # Denormalize
                true_mask = masks[i].cpu().numpy()
                pred_mask = preds[i].cpu().numpy()
                
                visualize_prediction(
                    image_np, 
                    true_mask, 
                    pred_mask, 
                    colormap
                )
                
                samples_shown += 1
                if samples_shown >= num_samples:
                    break


# Example usage:
# Uncomment to visualize predictions on validation set
# visualize_sample_predictions(model, val_loader, device, cityscapes_colormap, num_samples=3)

print("Visualization functions ready!")
print("Use visualize_sample_predictions() to see predictions with distinct colors.")


In [ ]:
def get_loss_fn(class_weights=None, ignore_index=255, label_smoothing=0.0):
    return nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=label_smoothing,
        ignore_index=ignore_index
    )


def get_scheduler(scheduler_type, optimizer, epochs):
    if scheduler_type == 'cosine':
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return None

def save_checkpoint(model, optimizer, scheduler, epoch, best_miou, checkpoint_path, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'best_miou': best_miou
    }
    torch.save(checkpoint, checkpoint_path)
    if is_best:
        best_path = checkpoint_path.parent / 'best_miou.pt'
        torch.save(checkpoint, best_path)

def train_epoch(model, dataloader, loss_fn, optimizer, device, scaler=None, scheduler=None):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(images)
                loss = loss_fn(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    if scheduler and isinstance(scheduler, torch.optim.lr_scheduler.CosineAnnealingLR):
        scheduler.step()

    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, loss_fn, device, num_classes, ignore_index):
    model.eval()
    total_loss = 0.0
    metrics = SegmentationMetrics(num_classes, ignore_index)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            metrics.update(preds.cpu(), masks.cpu())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, metrics.get_metrics()

def train_model(model, train_loader, val_loader, loss_fn, optimizer, scheduler,
                device, num_epochs, output_dir, use_amp=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    scaler = torch.amp.GradScaler() if use_amp else None
    best_miou = 0.0

    metrics_file = output_dir / 'metrics.csv'
    fieldnames = ['epoch', 'train_loss', 'val_loss', 'miou', 'pixel_acc'] + \
                 [f'class_{i}_iou' for i in range(config['num_classes'])]

    with open(metrics_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scaler, scheduler)
        
        val_loss, val_metrics = validate_epoch(
            model, val_loader, loss_fn, device, 
            config["num_classes"], config.get("ignore_index")
        )

        miou = val_metrics['miou']
        pixel_acc = val_metrics['pixel_acc']
        class_ious = val_metrics['class_ious']

        is_best = miou > best_miou
        if is_best:
            best_miou = miou

        checkpoint_path = output_dir / 'last.pt'
        save_checkpoint(model, optimizer, scheduler, epoch + 1, best_miou, checkpoint_path, is_best)

        metrics_row = {
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'miou': miou,
            'pixel_acc': pixel_acc
        }
        for i, iou in enumerate(class_ious):
            metrics_row[f'class_{i}_iou'] = iou

        with open(metrics_file, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerow(metrics_row)

        print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        print(f"mIoU: {miou:.4f}, Pixel Acc: {pixel_acc:.4f}")

    print(f"\nTraining completed! Best mIoU: {best_miou:.4f}")
    return best_miou

## Training Execution

In [14]:
# Setup training components
loss_fn = get_loss_fn(ignore_index=-100)  # PyTorch standard ignore value
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
scheduler = get_scheduler('cosine', optimizer, config['epochs'])

# Create output directory
output_dir = Path(config['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

# Save configuration
with open(output_dir / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Output directory: {output_dir}")

# Start training
best_miou = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=config['epochs'],
    output_dir=output_dir,
    use_amp=config['use_amp']
)

Output directory: /kaggle/working

Epoch 1/50


Train Loss: 1.0197, Val Loss: 0.5632
mIoU: 0.2827, Pixel Acc: 0.8579

Epoch 2/50


Train Loss: 0.6548, Val Loss: 0.4332
mIoU: 0.3116, Pixel Acc: 0.8782

Epoch 3/50


Train Loss: 0.5760, Val Loss: 0.3910
mIoU: 0.3321, Pixel Acc: 0.8849

Epoch 4/50


Train Loss: 0.5331, Val Loss: 0.3849
mIoU: 0.3436, Pixel Acc: 0.8882

Epoch 5/50


...

Train Loss: 0.2237, Val Loss: 0.3452
mIoU: 0.4542, Pixel Acc: 0.9141

Epoch 47/50


Train Loss: 0.2220, Val Loss: 0.3454
mIoU: 0.4570, Pixel Acc: 0.9144

Epoch 48/50


Train Loss: 0.2252, Val Loss: 0.3481
mIoU: 0.4528, Pixel Acc: 0.9146

Epoch 49/50


Train Loss: 0.2210, Val Loss: 0.3428
mIoU: 0.4527, Pixel Acc: 0.9143

Epoch 50/50


Train Loss: 0.2230, Val Loss: 0.3467
mIoU: 0.4523, Pixel Acc: 0.9142

Training completed! Best mIoU: 0.4570


## Model Evaluation and Visualization

In [15]:
# Load best model for evaluation
best_model_path = output_dir / 'best_miou.pt'
if best_model_path.exists():
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model with mIoU: {checkpoint['best_miou']:.4f}")

# Final evaluation on validation set
model.eval()
val_metrics = SegmentationMetrics(config["num_classes"], config.get("ignore_index"))

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final Evaluation"):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        val_metrics.update(preds, masks)

final_metrics = val_metrics.get_metrics()
print_metrics_table(final_metrics)

# Save final metrics
with open(output_dir / 'final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

Loaded best model with mIoU: 0.4570


Final Evaluation: 100%|██████████| 63/63 [00:06<00:00,  9.43it/s]


SEGMENTATION METRICS
mIoU: 0.4570
Pixel Accuracy: 0.9144

Per-Class IoU:
------------------------------
Class 0: 0.0000
Class 1: 0.9732
Class 2: 0.9430
Class 3: 0.9617
Class 4: 0.2006
Class 5: 0.1830
Class 6: 0.0639
Class 7: 0.9308
Class 8: 0.7225
Class 9: 0.3738
Class 10: 0.0836
Class 11: 0.8556
Class 12: 0.3417
Class 13: 0.3879
Class 14: 0.0000
Class 15: 0.3887
Class 16: 0.0000
Class 17: 0.3814
Class 18: 0.8917


## Inference Example

In [16]:
# Example inference on a single image
def predict_single_image(image_path, model, device, img_size):
    """Run inference on a single image."""
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    
    # Transform
    transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor()
    ])
    
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # Inference
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).squeeze(0)
    
    # Resize prediction back to original size
    pred_pil = T.ToPILImage()(pred.byte())
    pred_resized = pred_pil.resize(original_size, Image.NEAREST)
    
    return pred_resized

# Example usage (uncomment and modify path as needed)
# pred_mask = predict_single_image('path/to/your/image.jpg', model, device, config['img_size'])
# pred_mask.save('prediction.png')
# print("Prediction saved!")

print("Training and evaluation complete!")
print(f"Check {output_dir} for saved models and metrics.")

Training and evaluation complete!
Check /kaggle/working for saved models and metrics.
